In [9]:
import os
import warnings; warnings.filterwarnings("ignore")

import mysql.connector as conn
import numpy as np
import pandas as pd; pd.set_option("display.float_format","{:,.2f}".format)
import sqlalchemy
from dotenv import load_dotenv
from sqlalchemy.sql import text

In [10]:
# Takes environment variables from .env.
load_dotenv()

# Credentials for local PostgreSQL database.
pg_host = "localhost"
pg_database = "paid_content"
pg_schema = "user_profile"
pg_table = "activity"
pg_user = "postgres"
pg_pass = os.environ["POSTGRES_PASSWORD"]
pg_port = 5432

# Credentials for internal Delfi database.
dlf_name_articles = "delfi_articles"
dlf_name_paywall = "paywall"
dlf_host = os.environ["DELFI_DB_HOST"]
dlf_user = os.environ["DELFI_DB_USER"]
dlf_pass = os.environ["DELFI_DB_PASSWORD"]
dlf_port = os.environ["DELFI_DB_PORT"]

# MySQL connector.
delfi_paywall_conn = conn.connect(
    host=dlf_host,
    port=dlf_port,
    user=dlf_user,
    password=dlf_pass,
    database=dlf_name_paywall
)

# 90 days period

In [11]:
# Query all data about activity of paid content users from local PostgreSQL database.
postgresql_url = f"postgresql+psycopg2://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_database}"

engine = sqlalchemy.create_engine(postgresql_url)

activity_query = text("SELECT * FROM user_profile.activity WHERE user_delfi_id NOT IN (1415556, 963890, 930767)")

with engine.connect().execution_options(autocommit=True) as conn:
    activity_query_sql = conn.execute(activity_query)

initial_activity_data = pd.DataFrame(activity_query_sql.fetchall())

initial_activity_data["activity_date"] = initial_activity_data["timestamp"].dt.date
initial_activity_data["activity_year"] = initial_activity_data["timestamp"].dt.year
initial_activity_data["activity_month"] = initial_activity_data["timestamp"].dt.month

unique_users = initial_activity_data["user_delfi_id"].nunique()
num_row = len(initial_activity_data)
print(f"Initial data consists of {num_row} rows and {unique_users} unique users.")

Initial data consists of 438295 rows and 14150 unique users.


Here and in the subsequent calculations of average values I use median because distribution of all data is more or less skewed to the rigth. 

In [12]:
rfv_df = initial_activity_data.groupby("user_delfi_id").agg(
    frequency=("activity_date", lambda x: x.nunique()),
    volume=("article_id", lambda x: x.nunique()),
    last_visit=("activity_date", "max"),
    avg_active_time=("active_time", "median"),
    avg_scroll_depth=("scroll_depth", "median")
).reset_index()

# The latest date is literally the lastest date of data and is used to calculate recency.
latest_date = initial_activity_data["activity_date"].max().strftime("%Y-%m-%d")
anchor_date = pd.to_datetime(latest_date).date()
rfv_df["recency"] = (anchor_date - rfv_df["last_visit"]).dt.days

# Data with final RFV scores.
rfv_df["rfv_score"] = (rfv_df["frequency"] * np.sqrt((rfv_df["volume"]))) / (rfv_df["recency"] + 1)
rfv_df["user_delfi_id"] = rfv_df["user_delfi_id"].astype("string")
rfv_df.head()

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score
0,1735,3,3,2023-03-01,6.50,39.00,14,0.35
1,2140,43,74,2023-03-11,250.50,60.50,4,73.98
2,2320,37,51,2023-03-15,93.50,49.50,0,264.23
3,2872,8,10,2023-03-14,47.50,98.00,1,12.65
4,3069,34,70,2023-03-14,63.00,50.50,1,142.23


In [13]:
# User ID and User Delfi ID from Delfi internal database.
user_id_query = """
    SELECT id, delfi_id
    FROM paywall.users
    WHERE delfi_id IS NOT NULL
    """

user_id_data = pd.read_sql(user_id_query, delfi_paywall_conn)
user_id_data = user_id_data.astype("string")

In [14]:
rfv_df = rfv_df.merge(user_id_data, how="left", left_on="user_delfi_id", right_on="delfi_id").rename(columns={"id": "user_id"})
rfv_df.head()

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score,user_id,delfi_id
0,1735,3,3,2023-03-01,6.50,39.00,14,0.35,4362,1735
1,2140,43,74,2023-03-11,250.50,60.50,4,73.98,14449,2140
2,2320,37,51,2023-03-15,93.50,49.50,0,264.23,5856,2320
3,2872,8,10,2023-03-14,47.50,98.00,1,12.65,56979,2872
4,3069,34,70,2023-03-14,63.00,50.50,1,142.23,56747,3069


I tried to select just those users who were active during the whole period: theirs subscriptions valid_till date was greater than the upper bound of period and first purchase date was older than the lower bound of period. With the date of first purchase I eliminate those users who bought theirs first subscriptions anytime during the period. Hence in theory all users have the equal possibilities to visit website the same number of times and read the same amount of articles.

When time period is increasing the probability that some users would churn and buy subscription again after some break (aka returned user) is also increasing. Due to constraints of our database I cannot eliminate such behavior but in our case number of such users is quite small.

In [15]:
# Users who were active during the whole period.
users_query = """
    SELECT DISTINCT(P1.user_id)
    FROM (SELECT T.user_id, T.completion_timestamp, S.plan, US.renewal,
        ROW_NUMBER() OVER (PARTITION BY T.user_id ORDER BY T.completion_timestamp) AS row_num
        FROM paywall.transactions AS T
		LEFT JOIN paywall.user_subscriptions AS US
		ON T.user_id = US.user_id
		LEFT JOIN paywall.subscriptions AS S
		ON US.subscription_id = S.id
		WHERE US.service_id IN (1, 3, 4, 9, 10)
		AND S.included_in_count = 1
		AND T.completion_timestamp IS NOT NULL
		AND S.sf_category = "Paid subscriptions"
		AND US.valid_till > "2023-03-13 23:59:59") AS P1
    WHERE P1.row_num = 1
    AND P1.completion_timestamp < "2023-01-01 00:00:00"
    """

users = pd.read_sql(users_query, delfi_paywall_conn)
users = users.astype("string")

In [16]:
users_final_df = rfv_df.merge(users, on="user_id").drop(columns="delfi_id")
users_final_df = users_final_df.astype({"user_delfi_id": "string", "user_id": "string"})
users_final_df.head()

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score,user_id
0,2140,43,74,2023-03-11,250.50,60.50,4,73.98,14449
1,2320,37,51,2023-03-15,93.50,49.50,0,264.23,5856
2,2872,8,10,2023-03-14,47.50,98.00,1,12.65,56979
3,3069,34,70,2023-03-14,63.00,50.50,1,142.23,56747
4,11878,20,27,2023-03-15,119.00,74.00,0,103.92,99419


RFV score consist of three main metrics:
- Recency - number of days since user visited the website last time.
- Frequency - how much days user visited the website during the period.
- Volume - number of read articles (pageviews).

Since there are no correlations between RFV and other users related metrics like Time on page, Scroll depth, Count of transactions or Total amount spent, we cannot understand when RFV is good one or bad one. From business perspective we decided that users who visited our website in the last 7 days - are the good ones and others - not so good. The former we called FANS, and the latter - SLEEPERS. In the first table below it is visible that these two groups of users have very different metrics, so our theory is correct. 

In other words, we determined our threshold of recency and looking to the data we can find out average frequency and average volume when recency is equal to 6. Knowing these two metrics we can calculate RFV threshold using RFV formula.

In [23]:
recency_threshold = 6

syntetic_rfv_data = users_final_df.copy()
syntetic_rfv_data["user_type"] = np.where(users_final_df["recency"] <= recency_threshold, "fans", "sleepers")

pd.DataFrame({
    "Users": syntetic_rfv_data.groupby("user_type")["user_id"].count(),
    "Share": syntetic_rfv_data.groupby("user_type")["user_id"].count() / syntetic_rfv_data["user_id"].count() * 100,
    "RFV score": syntetic_rfv_data.groupby("user_type")["rfv_score"].median(),
    "Frequency": syntetic_rfv_data.groupby("user_type")["frequency"].median(),
    "Recency": syntetic_rfv_data.groupby("user_type")["recency"].median(),
    "Volume": syntetic_rfv_data.groupby("user_type")["volume"].median()
}).set_axis(["Fans: Recency equal or below 6", "Sleepers: Recency above 6"], axis="index")

,Users,Share,RFV score,Frequency,Recency,Volume
Fans: Recency equal or below 6,6493,61.97,39.66,18.00,1.00,25.00
Sleepers: Recency above 6,3985,38.03,0.26,3.00,21.00,3.00


In [24]:
frequency_threshold = syntetic_rfv_data[syntetic_rfv_data["recency"] == recency_threshold]["frequency"].median()
volume_threshold = syntetic_rfv_data[syntetic_rfv_data["recency"] == recency_threshold]["volume"].median()
print("Threshold of Frequency:", frequency_threshold)
print("Threshold of Volume:", volume_threshold)

Threshold of Frequency: 8.0
Threshold of Volume: 10.0


In [26]:
syntetic_rfv_score = (frequency_threshold * np.sqrt((volume_threshold))) / (recency_threshold + 1)
print("Threshold of RFV score:", syntetic_rfv_score)
print("Duration of period (Days):", initial_activity_data["activity_date"].nunique())

Threshold of RFV score: 3.614031611621005
Duration of period (Days): 74


# 28 days period